# U0.L3 — VAE on MNIST

**Flow-Based Generative Models · UFRJ · 2026.2** · lab notebook · date: see calendar-map

Today you train the first generative model of this course, and then *look* at it.
By the end you own **`vae-mnist-scratch`** — the codebase PS0 extends by swapping in
your U0.L4 U-Net encoder.

**How this lab works** (the standard set in U0.L1): numbered **TODO** cells are yours;
the instructor live-codes the first instance of each pattern. **Checkpoint cells**
appear every ~15 minutes — *if the assert passes, continue*. Solutions are released
after the session.

**Format note.** U0.L2 was scripts, because the object of study was the *run*. Today is
a notebook, because the object of study is *code you are still shaping* — and because
half the session is pictures. Training still uses the **hygiene stack** conventions from
U0.L2 (config, seed, MLflow tags, per-term logging).

Full prose, every figure and every number: the [U0.L3 lecture notes](../docs/U0/U0-L3-notes.html).

## Part A — Skeleton and decisions

*Reused in: PS0, directly — this is the codebase the problem set extends.*

Cell 0 of every notebook in this course does three jobs: prove the environment works,
fix the device, and seed **every** RNG stream (`random`, NumPy and torch keep
independent streams — seeding one of them is the classic way to be "seeded" and still
not reproducible).

In [ ]:
import math
import random
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

print(torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    """Seed all three RNG streams (python, numpy, torch incl. CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)

### The decision nobody writes down

U0.T3 wrote $p_\theta(x \mid z)$ and never committed to a family. You must, because
getting it wrong is **silent**:

- **Bernoulli decoder** on binary pixels — one probability per pixel, and
  $-\log p_\theta(x \mid z)$ is a binary cross-entropy.
- **Gaussian decoder** — a squared error, plus a $\sigma^2$ you either fix by hand
  (and silently reweight the ELBO) or learn (and watch run toward zero).

We choose **Bernoulli** — so the data must be binary, and MNIST is not. Binary
cross-entropy accepts fractional targets without complaint and computes something
perfectly well-defined that is **not the log-likelihood of any model**. The fix is
*dynamic binarization*: resample the image every time it is drawn.

In [ ]:
DATA_ROOT = "../scripts/python/data"   # torchvision downloads here if absent
N_VAL = 6_000                          # carved out of the 60k TRAIN split

tf = transforms.ToTensor()             # [0, 1] grayscale; binarization per draw
full = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tf)

g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(full), generator=g)
train_idx, val_idx = perm[:-N_VAL].tolist(), perm[-N_VAL:].tolist()

train_ds, val_ds = Subset(full, train_idx), Subset(full, val_idx)
print(f"train {len(train_ds)} | val {len(val_ds)} | test 10000 (untouched today)")


def binarize(x: torch.Tensor) -> torch.Tensor:
    """Dynamic binarization: a FRESH Bernoulli draw every time x is drawn."""
    return torch.bernoulli(x)

**The test split is not in this notebook.** We validate on 6 000 images held out of the
training split; the 10 000-image test split is read exactly once, later, by the
evaluation harness you build in U0.L4. That is the split discipline of this course, and
it is much easier to keep than to repair.

Look at what the decoder will actually be asked to model — the binarized draw, not the
grayscale original:

In [ ]:
xb = torch.stack([full[i][0] for i in val_idx[:8]])
fig, axes = plt.subplots(2, 8, figsize=(9, 2.4))
for k in range(8):
    axes[0, k].imshow(1 - xb[k, 0], cmap="gray"); axes[0, k].axis("off")
    axes[1, k].imshow(1 - binarize(xb)[k, 0], cmap="gray"); axes[1, k].axis("off")
axes[0, 0].set_title("grayscale", loc="left", fontsize=9)
axes[1, 0].set_title("one Bernoulli draw", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

### The config — an experiment is data, not code

Every quantity that can vary between runs lives here (U0.L2 hygiene stack). You will
train this model **twice** — `d_z = 2` for the latent pictures, `d_z = 16` for sample
quality — and the two runs differ by a **one-line config diff**.

Three settings are the U0.T2 recipe card being *obeyed*, not re-decided: AdamW at
$3\times10^{-4}$ with warmup–cosine; **GroupNorm, not BatchNorm** (the BN clause was
for bootcamp classifiers — this is a generative model); and near-zero-initialized
output heads (zero biases, small nonzero weights — see Checkpoint B for why the
weights are not zeroed too).

In [ ]:
@dataclass
class Config:
    run_name: str = "U0L3-dz2-s0"
    d_z: int = 2
    enc_width: int = 32
    dec_width: int = 64
    epochs: int = 15            # ~1 min on a laptop GPU; raise if you have time
    batch_size: int = 256
    lr: float = 3.0e-4          # U0.T2 recipe card
    weight_decay: float = 0.01
    warmup_frac: float = 0.03
    seed: int = 0
    tags: dict = field(default_factory=lambda: {"session": "U0.L3"})


cfg = Config()
cfg

## Part B — Encoder, decoder, reparameterization

### Why the head returns a logarithm

The encoder returns $\log \sigma_\varphi^2$, **never** $\sigma_\varphi$. A network head
is an unconstrained linear map — it can emit any real number — but $\sigma$ must be
positive. Parameterizing the logarithm makes *every* output legal and pushes positivity
into a single `exp` where it cannot be violated. A `softplus` or `relu` head can hand
you $\sigma = 0$ and then a division; clamping creates a region with exactly zero
gradient.

**The pattern, which recurs all course:** parameterize the *unconstrained* quantity and
transform.

In [ ]:
class Encoder(nn.Module):
    """x -> (mu_phi, logvar_phi), both (B, d_z)."""

    def __init__(self, d_z: int, width: int) -> None:
        super().__init__()
        w = width
        self.body = nn.Sequential(
            nn.Conv2d(1, w, 4, stride=2, padding=1),          # 28 -> 14
            nn.GroupNorm(8, w), nn.SiLU(),
            nn.Conv2d(w, 2 * w, 4, stride=2, padding=1),      # 14 -> 7
            nn.GroupNorm(8, 2 * w), nn.SiLU(),
            nn.Conv2d(2 * w, 4 * w, 3, stride=2, padding=1),  # 7 -> 4
            nn.GroupNorm(8, 4 * w), nn.SiLU(),
            nn.Flatten(),
        )
        # TODO(B.1, encoder part): two linear heads mapping the flattened body
        # output (4*w*16 features) to d_z each -- one for mu, one for logvar.
        # Then ZERO-INITIALISE both (weight and bias): the recipe card's
        # last-layer clause, and what makes the Checkpoint B assert meaningful.
        self.mu = ...
        self.logvar = ...
        raise NotImplementedError("TODO B.1")

    def forward(self, x):
        h = self.body(x)
        # TODO(B.1): return the pair (mu, logvar).
        raise NotImplementedError("TODO B.1")

### TODO B.2 — the trick, as one line

$$
z = \mu_\varphi(x) + \exp\!\big(\tfrac{1}{2}\log\sigma^2_\varphi(x)\big) \odot \varepsilon,
\qquad \varepsilon \sim \mathcal{N}(0, I_{d_z}).
$$

The entire derivation of the previous theory session collapses into one line with a
`randn_like` in it. Use `randn_like` rather than `randn(...)`: you can then never get
the shape or the device wrong.

In [ ]:
def reparameterize(mu, logvar):
    """z = mu + exp(logvar/2) * eps, eps ~ N(0, I). Shapes stay (B, d_z)."""
    # TODO(B.2, reparameterization part): one line.
    raise NotImplementedError("TODO B.2")

### TODO B.3 — the decoder returns logits

Return **logits**, not probabilities, and pair them with the *with-logits* form of
binary cross-entropy. It computes the stable branch instead of composing a `sigmoid`
with a `log` and losing the tails to floating point — the same rule as `log_softmax`
over `log(softmax(...))` from U0.T1, in its second costume.

In [ ]:
class Decoder(nn.Module):
    """z -> Bernoulli LOGITS over the 784 pixels, shape (B, 1, 28, 28)."""

    def __init__(self, d_z: int, width: int) -> None:
        super().__init__()
        w = width
        self.w = w
        self.proj = nn.Linear(d_z, 4 * w * 16)
        self.body = nn.Sequential(
            nn.GroupNorm(8, 4 * w), nn.SiLU(),
            nn.ConvTranspose2d(4 * w, 2 * w, 3, stride=2, padding=1),  # 4 -> 7
            nn.GroupNorm(8, 2 * w), nn.SiLU(),
            nn.ConvTranspose2d(2 * w, w, 4, stride=2, padding=1),      # 7 -> 14
            nn.GroupNorm(8, w), nn.SiLU(),
            nn.ConvTranspose2d(w, 1, 4, stride=2, padding=1),          # 14 -> 28
        )
        # TODO(B.3, decoder part): zero-init the LAST layer (recipe card).
        raise NotImplementedError("TODO B.3")

    def forward(self, z):
        # TODO(B.3): project, reshape to (B, 4w, 4, 4), run the body, return
        # the logits. Do NOT apply a sigmoid here.
        raise NotImplementedError("TODO B.3")

### Checkpoint B — is the latent standard normal at initialization?

If the assert passes, continue.

This is the zero-initialization clause paying out, with one amendment. The heads'
**biases** are zeroed and their **weights** are small but nonzero (std $10^{-3}$), so
$\mu_\varphi \approx 0$ and $\log\sigma^2_\varphi \approx 0$, hence
$\sigma_\varphi \approx 1$ and $z \approx \varepsilon$: the encoder starts
*essentially* as the prior, the KL starts near zero and has to be *earned*, and the
training curves in Part D stay readable as a story.

Zeroing the weights as well would make that exact — and would send **exactly zero**
gradient into the encoder trunk on the first optimizer step, which is the defect that
made the gradient check below unable to fail when this session was first prepared.

In [ ]:
set_seed(cfg.seed)
enc = Encoder(cfg.d_z, cfg.enc_width).to(DEVICE)
dec = Decoder(cfg.d_z, cfg.dec_width).to(DEVICE)

x_fixed = torch.stack([full[i][0] for i in val_idx[:64]]).to(DEVICE)
mu0, logvar0 = enc(binarize(x_fixed))
z0 = reparameterize(mu0, logvar0)

assert mu0.shape == (64, cfg.d_z) and logvar0.shape == (64, cfg.d_z)
assert z0.shape == (64, cfg.d_z)
assert dec(z0).shape == (64, 1, 28, 28)
assert mu0.abs().max() < 0.5, "mu head is not near zero at init"
assert logvar0.abs().max() < 0.5, "logvar head is not near zero at init"
print(f"|mu|max {mu0.abs().max():.3f}, |logvar|max {logvar0.abs().max():.3f}"
      "   <- small, because the head WEIGHTS are small (not zero)")

print(f"z mean {z0.mean():.3f}, std {z0.std():.3f}   <- should look like N(0, 1)")
print("Checkpoint B passed.")

The printed mean will not be exactly 0 — it is $128$ draws from $\mathcal{N}(0,1)$,
whose sample mean carries a standard error of $0.09$. The **assert** is on $\mu$ and
$\log\sigma^2$, which are exact; the printout is a sanity check. Knowing which of the
two to trust is itself the lesson.

## Part C — The ELBO in code

*Reused in: PS0 verbatim.*

$$
\mathcal{L}(\theta, \varphi; x)
= \underbrace{-\,\mathbb{E}_{\varepsilon \sim \mathcal{N}(0,I)}
   \big[\log p_\theta(x \mid z_\varphi(x,\varepsilon))\big]}_{\text{reconstruction}}
\;+\; \underbrace{\mathrm{KL}\big(q_\varphi(z \mid x)\,\big\|\,p(z)\big)}_{\text{regularization}}
$$

Both terms are computable. **The session's hardest bug is in how you add them up.**

### TODO C.1 — the reduction convention (the planted lesson)

> The loss is a **per-example** quantity: a log-likelihood of a whole image, and a KL
> between two distributions on the whole latent. Both are **sums over their own
> dimensions**. The batch is a Monte Carlo estimate of $\mathbb{E}_{x \sim q}$, so the
> batch — and only the batch — gets a **mean**.

Reduce the reconstruction by `mean` over pixels instead — which is PyTorch's *default* —
and you have divided that term by $784$ while leaving the KL alone. That is not a
rescaling of the loss. It is a **different objective**: a $\beta$-VAE with
$\beta = 784$, which nobody asked for, and whose loss curve still goes down.

In [ ]:
def recon_term(logits, x):
    """-log p_theta(x|z): SUM over the 784 pixels, MEAN over the batch."""
    per_pixel = F.binary_cross_entropy_with_logits(logits, x, reduction="none")
    # TODO(C.1, reconstruction part): reduce per_pixel (B, 1, 28, 28) to a
    # scalar -- sum over pixels FIRST, then mean over the batch.
    raise NotImplementedError("TODO C.1")

### TODO C.2 — the KL term, kept per dimension

For a diagonal Gaussian against $\mathcal{N}(0, I_{d_z})$ — quoted in U0.T3, derived in
UP.A, and **today it is a tool**:

$$
\mathrm{KL}\big(q_\varphi(z\mid x)\,\big\|\,p(z)\big)
= \frac{1}{2}\sum_{j=1}^{d_z}\Big(\mu_j^2 + \sigma_j^2 - 1 - \log \sigma_j^2\Big)
$$

**Return the `(B, d_z)` tensor and sum it at the call site.** Summing inside the
function would be tidier and would throw away exactly the information Part F needs:
which *individual* dimensions are doing work. Instrumentation you did not keep is
instrumentation you do not have.

In [ ]:
def kl_diag_gaussian(mu, logvar):
    """KL per dimension, per example -> (B, d_z). NOT summed."""
    # TODO(C.2, KL part): the closed form above, elementwise.
    # Careful: the formula uses log(sigma^2), which IS logvar -- no extra log.
    raise NotImplementedError("TODO C.2")

### TODO C.3 — a loss that returns its parts

> **Course rule — composite losses log their parts.** A single scalar cannot
> distinguish *"reconstruction improved"* from *"the KL collapsed"* — opposite events
> with the same effect on the total. Stated in U0.T2's recipe card; from here it is
> **enforced by the interface**.

This dictionary is part of the frozen `vae-mnist-scratch` contract that PS0 depends on.

In [ ]:
class VAE(nn.Module):
    """The `vae-mnist-scratch` contract (frozen at the end of this session):

        encode(x)                  -> (mu, logvar)
        reparameterize(mu, logvar) -> z
        decode(z)                  -> logits
        loss(x) -> {"loss", "recon", "kl", "kl_per_dim"}
    """

    def __init__(self, cfg: Config) -> None:
        super().__init__()
        self.cfg = cfg
        self.encoder = Encoder(cfg.d_z, cfg.enc_width)
        self.decoder = Decoder(cfg.d_z, cfg.dec_width)

    def encode(self, x):
        return self.encoder(x)

    @staticmethod
    def reparameterize(mu, logvar):
        return reparameterize(mu, logvar)

    def decode(self, z):
        return self.decoder(z)

    def loss(self, x):
        # TODO(C.3, assembly part): encode -> reparameterize -> decode, then
        # build the dict. kl_per_dim is the (B, d_z) tensor MEANED over the
        # batch only -- shape (d_z,) -- so Part F can read it per dimension.
        raise NotImplementedError("TODO C.3")

### Checkpoint C — the reduction, caught by arithmetic

If the assert passes, continue. The independent witness here is a number you can
compute in your head: at initialization the decoder outputs **logits of exactly zero**
(zero-init last layer), so every pixel is predicted with probability $0.5$, and

$$
-\log p_\theta(x \mid z) = 784 \times \log 2 \approx 543.4 \ \text{nats},
$$

whatever the image is. The KL is exactly $0$ for the same reason as Checkpoint B. If
your reconstruction reads $\approx 0.693$ instead of $\approx 543$, you reduced by
`mean` over pixels — that is the planted trap, caught before it can cost you an hour.

In [ ]:
set_seed(cfg.seed)
model = VAE(cfg).to(DEVICE)
with torch.no_grad():                  # a checkpoint, not a training step
    out = model.loss(binarize(x_fixed))

expected = 784 * math.log(2)
print(f"recon {out['recon']:.2f}  (expected {expected:.2f})   kl {out['kl']:.6f}")

assert abs(float(out["recon"]) - expected) < 1.0, (
    f"recon is {float(out['recon']):.3f}, expected ~{expected:.1f} nats/image. "
    "If it is ~0.693 you reduced by MEAN over pixels -- that is a beta-VAE "
    "with beta = 784, not this model."
)
assert float(out["kl"]) < 0.05, (
    "KL should start NEAR zero: the heads have small nonzero weights, so "
    "q_phi(z|x) starts close to the prior rather than equal to it."
)
assert out["kl_per_dim"].shape == (cfg.d_z,), "keep the per-dimension KL!"
print("Checkpoint C passed.")

### The gradient-check ritual, on a random loss

U0.L1's rule: verify a hand-written loss against finite differences in `float64`
**before** trusting a single training step, along **one fixed unit direction** $v$ —
two forward evaluations, whatever the tensor's size.

The ELBO adds a wrinkle: it is a *random* function, so differencing two independent
draws measures **noise**. Freeze $\varepsilon$ and the estimator becomes an ordinary
deterministic function of the parameters; reparameterization is exactly what makes
that legal, because the randomness is now an **input**.

Two traps, and the first one caught the preparation of this session:

1. **A check run at initialization cannot fail — it used to.** With the head
   *weights* zeroed, the gradient reaching any encoder *body* parameter is exactly
   zero at step 0: analytic zero, numeric zero, a perfect score from a measurement of
   nothing. The heads now carry small nonzero weights, which removes the cause. Take
   a few optimizer steps anyway, and **assert the analytic gradient is nonzero**
   before believing it agrees with anything.
2. **A check that has never failed is not known to work.** So make it fail on purpose:
   drop the $\tfrac{1}{2}$ from the KL — a plausible typo — and re-run.

Run the cell and read both verdicts.

In [ ]:
def gradient_check(corrupt: bool = False, n_warm: int = 30) -> dict:
    """Directional finite differences vs autograd, one tensor, float64."""
    set_seed(0)
    torch.set_default_dtype(torch.float64)
    m = VAE(cfg).to("cpu").double()
    x = torch.bernoulli(torch.rand(8, 1, 28, 28, dtype=torch.float64))
    eps = torch.randn(8, cfg.d_z, dtype=torch.float64)     # FIXED noise

    warm = torch.optim.AdamW(m.parameters(), lr=1e-3)      # trap 1: walk off zero
    for _ in range(n_warm):
        o = m.loss(x); warm.zero_grad(set_to_none=True)
        o["loss"].backward(); warm.step()

    def loss_at(w, p, kl_scale=0.5):
        with torch.no_grad():
            p.copy_(w.view_as(p))
        mu, logvar = m.encode(x)
        z = mu + torch.exp(0.5 * logvar) * eps
        r = recon_term(m.decode(z), x)
        k = (2 * kl_scale) * kl_diag_gaussian(mu, logvar).sum(dim=1).mean()
        return r + k

    p = m.encoder.body[0].weight
    base = p.detach().clone().flatten()
    m.zero_grad(set_to_none=True)
    loss_at(base, p, kl_scale=1.0 if corrupt else 0.5).backward()
    analytic = p.grad.detach().flatten().clone()
    assert float(analytic.norm()) > 1e-8, "analytic gradient is zero: vacuous check"

    h = 1e-6
    v = torch.randn_like(base); v /= v.norm()      # one fixed unit direction
    d_ad = float(analytic @ v)
    with torch.no_grad():            # finite differences on the CORRECT loss
        lo = float(loss_at(base - h * v, p))
        hi = float(loss_at(base + h * v, p))
    d_fd = (hi - lo) / (2 * h)
    atol, rtol = 1e-6, 1e-6          # atol floors float64 cancellation noise
    tol = atol + rtol * max(abs(d_ad), abs(d_fd))

    with torch.no_grad():            # leave the model as we found it
        p.copy_(base.view_as(p))
    torch.set_default_dtype(torch.float32)
    return {"d_ad": d_ad, "d_fd": d_fd, "gap": abs(d_ad - d_fd), "tol": tol,
            "accepted": abs(d_ad - d_fd) <= tol}


ok = gradient_check()
bad = gradient_check(corrupt=True)
print(f"correct loss     : d_AD {ok['d_ad']:.10e}  d_FD {ok['d_fd']:.10e}")
print(f"                   gap {ok['gap']:.2e}  (tol {ok['tol']:.2e})  "
      f"accepted={ok['accepted']}")
print(f"KL's 1/2 dropped : gap {bad['gap']:.2e}  (tol {bad['tol']:.2e})  "
      f"accepted={bad['accepted']}   <- must be False")
assert ok["accepted"] and not bad["accepted"]
print("The ritual passes, AND it is known to be able to fail.")

## Part D — Training with the hygiene stack

Nothing here is new machinery; that is the payoff of U0.L2. In your own course repo this
is `common/train.py` v2 doing the work — config in, per-term metrics out, MLflow tags,
warmup–cosine stepped **per step**. The wiring, verbatim:

```python
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fbgm-2026")            # ONE experiment per course phase
with mlflow.start_run(run_name=cfg.run_name):
    mlflow.log_params(flatten(cfg))
    mlflow.set_tags({"session": "U0.L3", "tag": f"dz{cfg.d_z}", "seed": cfg.seed})
    trainer.fit(train_loader, val_loader, epochs=cfg.epochs)
```

**TODO D.1 (logging part).** Add two per-epoch `mlflow.log_figure` calls: a
**reconstruction grid** on a *fixed* validation batch, and a **prior-sample grid** from
a *fixed* bank of $\varepsilon$ draws. Both "fixed" are load-bearing — a fresh draw each
epoch gives a slideshow in which improvement and reshuffling look identical. The same
trick returns for sample panels in the U3 labs.

In [ ]:
def warmup_cosine(step, total, warmup):
    """Multiplier on the base LR -- the U0.T2 default, stepped PER STEP."""
    if step < warmup:
        return step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * p))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    kl_dim = torch.zeros(model.cfg.d_z, device=DEVICE)
    n = 0
    for x, _ in loader:
        o = model.loss(binarize(x.to(DEVICE)))
        for k in tot:
            tot[k] += float(o[k])
        kl_dim += o["kl_per_dim"]; n += 1
    model.train()
    return {**{k: v / n for k, v in tot.items()},
            "kl_per_dim": (kl_dim / n).cpu().numpy()}

In [ ]:
FIXED_EPS = torch.randn(64, cfg.d_z)     # the fixed noise bank -- draw ONCE


def log_epoch_panels(model, epoch):
    """Two panels per epoch, both from FIXED inputs so epochs are comparable."""
    # TODO(D.1, logging part): build two figures and (in your own repo) pass
    # each to mlflow.log_figure(fig, f"recon_epoch{epoch}.png") etc.
    #   1. reconstructions of `x_fixed` (sigmoid of the decoded logits),
    #   2. prior samples decoded from FIXED_EPS.
    # Return them as a (fig_recon, fig_samples) pair.
    raise NotImplementedError("TODO D.1")

In [ ]:
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          drop_last=True, num_workers=2,
                          generator=torch.Generator().manual_seed(cfg.seed))
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=2)

set_seed(cfg.seed)
model = VAE(cfg).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                        weight_decay=cfg.weight_decay)
total = cfg.epochs * len(train_loader)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: warmup_cosine(s, total, int(cfg.warmup_frac * total)))

history = {"epoch": [], "recon": [], "kl": []}
for epoch in range(1, cfg.epochs + 1):
    for x, _ in train_loader:
        o = model.loss(binarize(x.to(DEVICE)))
        opt.zero_grad(set_to_none=True)
        o["loss"].backward()
        opt.step(); sched.step()
    val = evaluate(model, val_loader)
    history["epoch"].append(epoch)
    history["recon"].append(val["recon"])
    history["kl"].append(val["kl"])
    print(f"epoch {epoch:2d}  loss {val['loss']:8.3f}  "
          f"recon {val['recon']:8.3f}  kl {val['kl']:6.3f}")

### Read the curves

The reconstruction term falls steeply. The KL term **rises from zero** — and it rises
even though the objective is *minimizing* a sum containing it. Nothing is broken: the KL
is the price the model pays for an informative latent, and reconstruction is willing to
pay it. The encoder started *equal* to the prior (Checkpoint B) and buys information
from it only as far as reconstruction pays.

**If your KL curve instead falls to zero, you are looking at the reduction trap.**

*A note on digits.* The figures and numbers in the lecture notes come from
`scripts/python/fig_u0_l3_vae.py`, whose loop consumes randomness in a slightly
different order than this notebook's. Your last decimal places will differ (here:
recon $\approx 142.4$, KL $\approx 6.0$; the notes report $142.6$ and $6.02$) and
nothing is wrong. Anything that has to match *exactly* is an `assert`, not a printout.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3))
a.plot(history["epoch"], history["recon"], color="#003366")
a.set_title("reconstruction (nats/image)"); a.set_xlabel("epoch")
b.plot(history["epoch"], history["kl"], color="#d97a1f")
b.set_title("KL (nats/image)"); b.set_xlabel("epoch")
for ax in (a, b):
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

## Part E — The latent space

The payoff, and it needs `d_z = 2`, because two dimensions can be drawn.

Encode the validation set to its posterior **means** and colour by digit label. Nothing
in the loss mentions a class — the labels are painted on *afterwards*. The only pressure
is that the decoder must rebuild $x$ from $z$, so images that look alike must land near
each other. And notice where the cloud sits: the KL is pulling the aggregate of all the
$q_\varphi(z\mid x)$ toward $\mathcal{N}(0, I)$, and it *has* to succeed, because
generation samples from the prior.

In [ ]:
@torch.no_grad()
def encode_split(model, loader):
    model.eval()
    mus, ys = [], []
    for x, y in loader:
        mu, _ = model.encode(binarize(x.to(DEVICE)))
        mus.append(mu.cpu().numpy()); ys.append(y.numpy())
    model.train()
    return np.concatenate(mus), np.concatenate(ys)


mu_val, y_val = encode_split(model, val_loader)

fig, ax = plt.subplots(figsize=(5.4, 5))
for d in range(10):
    m = y_val == d
    ax.scatter(mu_val[m, 0], mu_val[m, 1], s=3, alpha=.55, label=str(d), lw=0)
th = np.linspace(0, 2 * np.pi, 200)
for r in (1, 2):
    ax.plot(r * np.cos(th), r * np.sin(th), color="#6b7280", lw=1, alpha=.8)
ax.set_xlabel(r"$\mu_\varphi(x)_1$"); ax.set_ylabel(r"$\mu_\varphi(x)_2$")
ax.set_aspect("equal"); ax.legend(markerscale=3, fontsize=8, ncol=2,
                                  loc="center left", bbox_to_anchor=(1.01, .5))
plt.tight_layout(); plt.show()

### The decoder as a map of the plane

Evaluate the decoder on a lattice of the latent plane — at **Gaussian quantiles**, not
uniform spacing, so the grid samples the prior evenly. No data is involved in making
this picture.

This is the figure to carry out of the session: the decoder has turned a featureless
Gaussian plane into an organized atlas of digits, and it is *continuous*.

In [ ]:
from scipy.stats import norm as scipy_norm

n = 16
qs = scipy_norm.ppf(np.linspace(0.02, 0.98, n))
zz = np.stack(np.meshgrid(qs, -qs, indexing="xy"), axis=-1).reshape(-1, 2)
with torch.no_grad():
    model.eval()
    sheet = torch.sigmoid(
        model.decode(torch.tensor(zz, dtype=torch.float32, device=DEVICE))
    )[:, 0].cpu().numpy()
    model.train()

canvas = np.ones((n * 28, n * 28))
for k in range(n * n):
    r, c = divmod(k, n)
    canvas[r * 28:(r + 1) * 28, c * 28:(c + 1) * 28] = sheet[k]

plt.figure(figsize=(5.6, 5.6))
plt.imshow(1 - canvas, cmap="gray"); plt.xticks([]); plt.yticks([])
plt.xlabel(r"$z_1$ (Gaussian quantiles)"); plt.ylabel(r"$z_2$ (Gaussian quantiles)")
plt.tight_layout(); plt.show()

### TODO E.1 — walking between two digits

Encode two validation images, interpolate between their $\mu_\varphi$, decode the path.

The standard advice is **slerp** for Gaussian latents, because the straight line between
two codes dips toward the origin, through a low-norm region the decoder rarely saw. Half
of that is right, and the honest version is worth having: for $z \sim \mathcal{N}(0, I_d)$
the norm $\lVert z \rVert$ concentrates — its mean grows like $\sqrt{d}$ while its
standard deviation stays near $1/\sqrt{2}$ *for every $d$*. So the deficit has to be
measured in units of that spread, and at $d_z = 2$ it is small enough that the choice
barely matters. Implement both, decode both, and see for yourself.

**Expect your two numbers to disagree with the notes' table, and expect the *sign* to
be unstable at $d_z = 2$.** With only two dimensions the deviation depends on which
pair of images you picked and on the run; the notes' own $d_z = 2$ row came out with
lerp *below* typical, this notebook's run puts it slightly above. That instability is
the finding — the concentration argument for slerp is a **high-dimensional** one, and
$d_z = 2$ is the wrong place to look for it. The $d_z = 16$ row is the robust one.

In [ ]:
def slerp(z0, z1, ts):
    """Spherical interpolation between two latents, at parameters ts."""
    # TODO(E.1, interpolation part): with Omega the angle between z0 and z1,
    #   slerp(t) = sin((1-t)Omega)/sin(Omega) * z0 + sin(t Omega)/sin(Omega) * z1
    # Guard the near-parallel case (Omega ~ 0) by falling back to lerp.
    raise NotImplementedError("TODO E.1")


def lerp(z0, z1, ts):
    return np.outer(1 - ts, z0) + np.outer(ts, z1)

In [ ]:
i0 = int(np.where(y_val == 0)[0][0])
i1 = int(np.where(y_val == 1)[0][0])
z0, z1 = mu_val[i0].astype(float), mu_val[i1].astype(float)
ts = np.linspace(0, 1, 9)

fig, axes = plt.subplots(2, 9, figsize=(9, 2.4))
for row, (path, name) in enumerate([(slerp(z0, z1, ts), "slerp"),
                                    (lerp(z0, z1, ts), "lerp")]):
    with torch.no_grad():
        model.eval()
        imgs = torch.sigmoid(model.decode(
            torch.tensor(path, dtype=torch.float32, device=DEVICE)))[:, 0]
        model.train()
    for k in range(9):
        axes[row, k].imshow(1 - imgs[k].cpu().numpy(), cmap="gray")
        axes[row, k].axis("off")
    axes[row, 0].set_title(name, loc="left", fontsize=9)
plt.tight_layout(); plt.show()

# The measurement that actually decides it: how many standard deviations of the
# PRIOR'S OWN norm distribution each path's midpoint sits away from typical.
from math import lgamma, exp, sqrt
d = cfg.d_z
mean_r = sqrt(2) * exp(lgamma((d + 1) / 2) - lgamma(d / 2))
sd_r = sqrt(max(d - mean_r ** 2, 1e-12))
for name, path in [("slerp", slerp(z0, z1, ts)), ("lerp", lerp(z0, z1, ts))]:
    mid = np.linalg.norm(path[len(ts) // 2])
    print(f"{name:6s} midpoint |z| = {mid:.3f}  "
          f"({(mid - mean_r) / sd_r:+.2f} sd from the prior's typical norm)")

## Part F — Posterior collapse, in passing

Five minutes. Pull up the per-dimension KL you carefully kept in TODO C.2.

The story you may have been told is that with $d_z = 16$ several dimensions' KL sits at
essentially zero, ignored by the decoder. **Measure it before believing it.**

In [ ]:
final = evaluate(model, val_loader)
kl_dim = np.sort(final["kl_per_dim"])[::-1]
COLLAPSE_NATS = 0.01

plt.figure(figsize=(5, 3))
plt.bar(range(len(kl_dim)), np.maximum(kl_dim, 1e-4),
        color=["#9aa2aa" if v < COLLAPSE_NATS else "#003366" for v in kl_dim])
plt.axhline(COLLAPSE_NATS, color="#d97a1f", ls="--")
plt.yscale("log"); plt.xlabel("latent dimension (sorted)"); plt.ylabel("KL (nats)")
plt.tight_layout(); plt.show()

print(f"dimensions below {COLLAPSE_NATS} nats: "
      f"{int((kl_dim < COLLAPSE_NATS).sum())} of {len(kl_dim)}")
print(f"quietest dimension carries {kl_dim.min():.3f} nats")

### What the measurement says

With `d_z = 2` there is nothing to collapse. Re-run the notebook with `d_z = 16` (one
config line) and you will find **it still does not happen**: on this model, at this
width, not one of the sixteen dimensions is unused — the quietest carries about
$1.06$ nats, a hundred times the threshold.

The phenomenon is real, but it needs latent capacity *in excess of what the decoder can
use*. Sweeping $d_z$ with everything else fixed (the numbers below are from the session
figures, and you can reproduce any row by editing one config line):

| $d_z$ | dead (<0.01 nats) | nearly dead (<0.1) | **total KL** | reconstruction |
|---|---|---|---|---|
| 16 | 0 | 0 | 24.6 | 73.1 |
| 32 | 0 | 0 | **30.1** | 68.5 |
| 64 | 2 | 31 | **30.5** | 69.3 |
| 128 | 34 | 99 | **30.3** | 70.0 |
| 64, decoder 2.5× wider | 10 | 32 | 30.2 | 66.9 |

**Total KL saturates at about $30$ nats and stays there.** The model has an information
budget set by what its decoder can use; beyond that point extra dimensions are not used
*badly* — they are not used at all. Posterior collapse is what surplus capacity looks
like from the inside.

**The mechanism.** If the decoder reconstructs just as well ignoring coordinate $j$,
then $j$ earns nothing in the reconstruction term — and the KL is a *cost*, minimized at
$q_\varphi(z_j \mid x) = p(z_j)$. The encoder switches it off. Not pathological
optimization: **the objective, minimized correctly.** Mitigations exist (KL annealing,
free bits) and are named here, not implemented.

**And the moral of the whole session:** the aggregate KL of the $d_z = 64$ run is $30.5$
nats — indistinguishable from the $d_z = 32$ run's $30.1$ — while half of one model is
inert. You could only ever see that because you logged the KL **per dimension**. The
instrument was one line in TODO C.2, *kept* rather than summed away.

## Freeze `vae-mnist-scratch`

```python
encode(x)                  -> (mu, logvar)     # PS0 replaces THIS and nothing else
reparameterize(mu, logvar) -> z
decode(z)                  -> logits
loss(x) -> {"loss", "recon", "kl", "kl_per_dim"}
```

Reconstruction is summed over pixels; KL is summed over dimensions; both are averaged
over the batch. Data is dynamically binarized. **Commit it and tag `u0l3-freeze`.**

**PS0**, in outline (assigned formally at the close of U0.L5): take this VAE, replace
the encoder with your **`unet-skeleton`** (U0.L4), train it with the **`hygiene-stack`**
(U0.L2), and evaluate it with your **`eval-harness`** (U0.L4) — FID at the
course-standard sample count, plus panels. Graded on **correctness and
reproducibility**, not on how good the samples look.

**Next session (U0.T4):** architectures for the road — attention, U-Net anatomy, and
time/positional embeddings, the part every model after this point is built from.